# coerce-float-arg-to-array — worked example 1: coerce_scalar: promote real int/float, pass through complex and bool

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `coerce-float-arg-to-array`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

In an autograd wrapper, Python scalar operands (e.g. `multiply(t, 3.0)`) must be promoted to 0-D tensors so back fns can do tensor math without type checks. The traps are that **`bool` is a subclass of `int`** (it must NOT be coerced — it is a control flag), and that `complex` is a distinct numeric type the raw torch op may want left alone. The rule promotes only real `int`/`float` to `torch.tensor(float(arg))` and passes everything else through.

## Worked solution

**Step 1 — guard `bool` first.** `isinstance(True, int)` is `True` in Python, so if we checked `(int, float)` before `bool`, flags like `keepdim=True` would be wrongly wrapped. We return early for `bool`.

**Step 2 — guard `complex`.** `complex` is numeric but not `int`/`float`, so the `(int, float)` branch already won't catch it. We make that explicit by passing it through — promoting it via `float()` would actually raise `TypeError: can't convert complex to float`, so passing through is both correct and safe.

**Step 3 — coerce real scalars.** Only now do we test `(int, float)`. We call `float(arg)` so even an `int` like `5` becomes `5.0`, then wrap in `t.tensor(...)`, giving a 0-D `float32` tensor (the default dtype for `t.tensor(5.0)`).

**Step 4 — pass-through default.** Tensors, ndarrays, tuples, `None`, strings all fall through to `return arg` unchanged — no copy, identity preserved.

**Why it works:** ordering the `bool` and `complex` guards before the `(int, float)` check is what makes the coercion total and trap-free; everything that is not a plain real scalar is left exactly as the downstream unbox stage expects it.

In [ ]:
def coerce_scalar(arg):
    # bool is a subclass of int — control flag, never coerce.
    if isinstance(arg, bool):
        return arg
    # complex is numeric but float() would raise on it; leave it raw.
    if isinstance(arg, complex):
        return arg
    if isinstance(arg, (int, float)):
        return t.tensor(float(arg))
    return arg

# Exercise it on a mix of types.
r_float = coerce_scalar(2.5)
r_int = coerce_scalar(7)
r_bool = coerce_scalar(True)
r_complex = coerce_scalar(1 + 2j)
r_tensor = coerce_scalar(t.ones(3))
print(r_float, r_float.dtype, r_float.ndim)
print(r_int, r_int.dtype)
print(r_bool, type(r_bool).__name__)
print(r_complex, type(r_complex).__name__)
print(r_tensor.shape, r_tensor is coerce_scalar(r_tensor))